# Module 6: AI Gateways and MLOps for LLM Deployment

Build a centralized gateway for model routing, cost control, and provider failover.


## 1. Setup

Load the gateway core and sample inputs.


In [ ]:
import sys
from pathlib import Path

module_dir = Path.cwd()
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

from ai_gateway_core import AIGateway, sample_receipts

print("✓ Setup complete")


## 2. Initialize Gateway

This simulates a centralized AI gateway (LiteLLM/Kong style).


In [ ]:
gateway = AIGateway()
receipts = sample_receipts()

print("✓ Gateway initialized")
print("Routes:", gateway.primary_route)


### Inspect Sample Receipts


In [ ]:
receipts


## 3. Smart Routing

- simple -> low-cost model
- moderate/complex -> stronger models


In [ ]:
for rid, receipt in receipts.items():
    result = gateway.route_and_infer(receipt)
    print(rid, "->", result.provider, result.model, "cost", result.estimated_cost_usd)


## 4. Fallback Simulation

Force provider outages and verify automatic rerouting.


In [ ]:
# simulate OpenAI outage
gateway.set_provider_health("openai", False)

complex_receipt = receipts["COMPLEX-DINNER"]
result_failover_1 = gateway.route_and_infer(complex_receipt)
print("OpenAI down ->", result_failover_1.to_dict())

# simulate Azure outage too
gateway.set_provider_health("azure", False)
result_failover_2 = gateway.route_and_infer(complex_receipt)
print("OpenAI + Azure down ->", result_failover_2.to_dict())

# bring providers back
gateway.set_provider_health("openai", True)
gateway.set_provider_health("azure", True)


## 5. Logging and Cost Visibility

Centralized logs and cumulative cost are key gateway benefits.


In [ ]:
print("Total logged events:", len(gateway.logs))
print("Total estimated cost USD:", gateway.total_cost())

for row in gateway.logs[-6:]:
    print(row)


## 6. Build Artifact: gateway_config.yaml

Generate the production-style gateway routing/failover config.


In [ ]:
cfg = gateway.generate_gateway_config_yaml()
print(cfg)

Path("gateway_config.yaml").write_text(cfg)
print("✓ Wrote gateway_config.yaml")


## 7. Assertions (Smoke Checks)


In [ ]:
# Simple receipt should route to cheap provider by default.
g2 = AIGateway()
simple = g2.route_and_infer(receipts["SIMPLE-COFFEE"])
assert simple.provider == "groq"

# Complex fallback chain should avoid unhealthy primary.
g3 = AIGateway()
g3.set_provider_health("openai", False)
fallback = g3.route_and_infer(receipts["COMPLEX-DINNER"])
assert fallback.provider in {"azure", "anthropic"}

print("✓ Module 6 gateway smoke checks passed")
